In [67]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
# Helper functions
def clean_cell(text):
    # text=td.get_text(strip=True)
    if text.upper() in ['N/A', 'NA', '-', '','+']:
        return None
    if text.startswith("+"):
        text=text[1:]
    clean_num= text.replace(',', '')
    if clean_num.replace('.', '',1).isdigit():
        return float(clean_num) if '.' in clean_num else int(clean_num)
    return text
# Decalring the headers and url
url ="https://www.worldometers.info/coronavirus/"
headers={"User-agents":"Mozzila/5"}
response=requests.get(url, headers)
# declaring the list to hold the headers
headers_list=[]
rows_data=[]
# checking the repsonse is working or not
print("Processing file...")
if response.status_code==200:
    # create or connect the contents with the soup
    soup=BeautifulSoup(response.text, 'html.parser')
    # find a specific content using soup
    table=soup.find('table', class_="table")
    # print(table) 
    # Getting the first headers using head tag
    thead=table.find('thead')
    # Getting each table headers using comprehesnion
    headers_list=[th.get_text(strip=True) for th in thead.find_all('th')]
    # print(headers_list)
    tbody=table.find('tbody')
    # print(tbody)
    for tr in tbody.find_all('tr'):
        td_tags =tr.find_all('td')
        if not td_tags:
            continue
        row_cells=[clean_cell(td.get_text(strip=True)) for td in td_tags]
        rows_data.append(row_cells)
        # print(td_tags)
    # print(rows_data)
          # print(tr)
    output_file_name="corona_virus_file.csv"  
    df=pd.DataFrame(rows_data, columns=headers_list)
    df.dropna(how='all', inplace=True)
    df.to_csv(output_file_name, encoding='utf-8')
    print(f"{len(df) }countries file saved to your desktop")
    print(df.columns)
    high_death=df[df['TotalDeaths']>1000000]
    print(f"Total death more than 1000000 is {high_death}")
    
else:
    print("Failed to connect with the website, please try agian!")


Processing file...
239countries file saved to your desktop
Index(['#', 'Country,Other', 'TotalCases', 'NewCases', 'TotalDeaths',
       'NewDeaths', 'TotalRecovered', 'NewRecovered', 'ActiveCases',
       'Serious,Critical', 'Tot Cases/1M pop', 'Deaths/1M pop', 'TotalTests',
       'Tests/1M pop', 'Population', 'Continent', '1 Caseevery X ppl',
       '1 Deathevery X ppl', '1 Testevery X ppl', 'New Cases/1M pop',
       'New Deaths/1M pop', 'Active Cases/1M pop'],
      dtype='object')
Total death more than 1000000 is      #  Country,Other  TotalCases  NewCases  TotalDeaths  NewDeaths  \
0  NaN  North America   131889132       NaN    1695941.0        NaN   
1  NaN           Asia   221500265       NaN    1553662.0        NaN   
2  NaN         Europe   253406198       NaN    2101824.0        NaN   
3  NaN  South America    70200879       NaN    1367332.0        NaN   
7  NaN          World   704753890       0.0    7010681.0        0.0   
8  1.0            USA   111820082       NaN    121